# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shahzaib-Ali59/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
%pip -q install duckdb huggingface_hub

import os
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':  f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':  f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':   f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:15} {n:>12,} rows')

dim_clients              104 rows
dim_content          519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily        78,835,655 rows
fact_query_90d     2,414,248 rows


In [16]:
print("=== dim_content columns ===")
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']} LIMIT 1").df())

print("\n=== fact_daily columns ===")
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily']} LIMIT 1").df())

=== dim_content columns ===
                   column_name column_type null   key default extra
0               client_hash_id     VARCHAR  YES  None    None  None
1              content_hash_id     VARCHAR  YES  None    None  None
2              keyword_hash_id     VARCHAR  YES  None    None  None
3                  url_hash_id     VARCHAR  YES  None    None  None
4           keyword_char_count      BIGINT  YES  None    None  None
5          keyword_token_count      BIGINT  YES  None    None  None
6               url_char_count      BIGINT  YES  None    None  None
7         content_created_date        DATE  YES  None    None  None
8         content_updated_date        DATE  YES  None    None  None
9                 content_type     VARCHAR  YES  None    None  None
10               search_volume      BIGINT  YES  None    None  None
11                 competition      DOUBLE  YES  None    None  None
12           competition_level     VARCHAR  YES  None    None  None
13                  

In [17]:
signal1 = con.sql(f"""
    WITH content_march AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS impressions_march,
               SUM(gsc_clicks) AS clicks_march
        FROM {TABLES['fact_daily']}
        WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
        GROUP BY 1
        HAVING SUM(gsc_impressions) >= 50
    ),
    joined AS (
        SELECT c.content_hash_id,
               DATE '2026-03-31' - c.content_updated_date AS days_since_update,
               m.impressions_march, m.clicks_march,
               CAST(m.clicks_march AS DOUBLE) / m.impressions_march AS ctr
        FROM content_march m
        JOIN {TABLES['dim_content']} c ON m.content_hash_id = c.content_hash_id
        WHERE c.is_published IS TRUE AND c.is_deleted IS NOT TRUE
          AND c.content_updated_date IS NOT NULL
    )
    SELECT
        CASE
            WHEN days_since_update <= 30 THEN '0-30d'
            WHEN days_since_update <= 90 THEN '31-90d'
            WHEN days_since_update <= 180 THEN '91-180d'
            WHEN days_since_update <= 365 THEN '181-365d'
            ELSE '365d+'
        END AS staleness_bucket,
        COUNT(*) AS n,
        ROUND(AVG(ctr), 4) AS avg_ctr
    FROM joined
    GROUP BY 1
    ORDER BY MIN(days_since_update)
""").df()

print(signal1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  staleness_bucket      n  avg_ctr
0            0-30d  95982   0.0027
1           31-90d  19878   0.0020
2          91-180d    182   0.0046
3         181-365d     21   0.0021


In [18]:
signal2 = con.sql(f"""
    WITH content_march AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS impressions_march,
               SUM(gsc_clicks) AS clicks_march,
               AVG(gsc_avg_position) AS avg_position
        FROM {TABLES['fact_daily']}
        WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
        GROUP BY 1
        HAVING SUM(gsc_impressions) >= 50
    )
    SELECT
        CASE
            WHEN avg_position <= 3 THEN '1-3'
            WHEN avg_position <= 10 THEN '4-10'
            WHEN avg_position <= 20 THEN '11-20'
            ELSE '20+'
        END AS position_bucket,
        COUNT(*) AS n,
        ROUND(AVG(CAST(clicks_march AS DOUBLE) / impressions_march), 4) AS avg_ctr
    FROM content_march
    GROUP BY 1
    ORDER BY MIN(avg_position)
""").df()

print(signal2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  position_bucket      n  avg_ctr
0             1-3   9687   0.0037
1            4-10  52218   0.0033
2           11-20  24294   0.0024
3             20+  29915   0.0013


## 1. Signal checks

**Signal 1 — Staleness (behind FlyRank's refresh flags):** Bucketed content_updated_date
into staleness ranges, checked average CTR per bucket (March 2026, pages with 50+ impressions).

**Verdict: MIXED.** Large-sample buckets (0-30d, n=95,982; 31-90d, n=19,878) show a mild
expected decline in CTR as staleness increases. But small-sample buckets (91-180d, n=182;
181-365d, n=21) show the opposite pattern — likely noise given how few rows they contain.
No pages in this slice are stale beyond 365 days, which is itself a data limit worth naming.
I don't trust staleness alone as a strong signal here; it needs a larger, more balanced
sample before I'd rely on it in isolation.

**Signal 2 — CTR vs. position (behind FlyRank's CTR-fix logic):** Bucketed average March
position, checked average CTR per bucket.

**Verdict: CONFIRMED.** CTR falls monotonically as position worsens (0.0037 → 0.0033 →
0.0024 → 0.0013), across large, comparable sample sizes in every bucket (9,687 to 52,218
rows). This is a real, trustworthy signal to build a rule on.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The rule

Given Signal 2's strength, my rule flags pages that rank reasonably well (so fixing them is
worth the effort) but have a CTR meaningfully below what their position bucket would predict
— a "CTR underperformance" signal, exactly the CTR-fix logic from the session.

**Score:** gap between a page's actual CTR and its position-bucket's average CTR (negative = underperforming).
**Reason code:** `CTR_BELOW_POSITION_BENCHMARK`
**Action label:** `review_meta_title` for flagged pages, `monitor` otherwise.

In [20]:
scored = con.sql(f"""
    WITH content_march AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS impressions_march,
               SUM(gsc_clicks) AS clicks_march,
               AVG(gsc_avg_position) AS avg_position
        FROM {TABLES['fact_daily']}
        WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
        GROUP BY 1
        HAVING SUM(gsc_impressions) >= 50
    ),
    with_ctr AS (
        SELECT *,
               CAST(clicks_march AS DOUBLE) / impressions_march AS ctr,
               CASE
                   WHEN avg_position <= 3 THEN '1-3'
                   WHEN avg_position <= 10 THEN '4-10'
                   WHEN avg_position <= 20 THEN '11-20'
                   ELSE '20+'
               END AS position_bucket
        FROM content_march
    ),
    bucket_avg AS (
        SELECT position_bucket, AVG(ctr) AS bucket_avg_ctr
        FROM with_ctr GROUP BY 1
    )
    SELECT w.content_hash_id, w.impressions_march, w.clicks_march, w.avg_position,
           w.ctr, w.position_bucket, b.bucket_avg_ctr,
           w.ctr - b.bucket_avg_ctr AS action_score,
           'CTR_BELOW_POSITION_BENCHMARK' AS reason_code,
           CASE WHEN w.ctr < b.bucket_avg_ctr THEN 'review_meta_title' ELSE 'monitor' END AS action
    FROM with_ctr w
    JOIN bucket_avg b ON w.position_bucket = b.position_bucket
    ORDER BY action_score ASC, impressions_march DESC
""").df()

import os
os.makedirs('work/outputs', exist_ok=True)
scored.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"{len(scored):,} rows written to work/outputs/baseline_action_score.csv")
scored.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

116,114 rows written to work/outputs/baseline_action_score.csv


,content_hash_id,impressions_march,clicks_march,avg_position,ctr,position_bucket,bucket_avg_ctr,action_score,reason_code,action
0,content_fa17add7836d36c3,12588.0,0.0,1.902457,0.0,1-3,0.003682,-0.003682,CTR_BELOW_POSITION_BENCHMARK,review_meta_title
1,content_d397987113cb84a0,9887.0,0.0,1.953104,0.0,1-3,0.003682,-0.003682,CTR_BELOW_POSITION_BENCHMARK,review_meta_title
2,content_a27b382f00aa75c6,7736.0,0.0,2.259812,0.0,1-3,0.003682,-0.003682,CTR_BELOW_POSITION_BENCHMARK,review_meta_title
3,content_83167156f76e33e5,6827.0,0.0,1.136714,0.0,1-3,0.003682,-0.003682,CTR_BELOW_POSITION_BENCHMARK,review_meta_title
4,content_1bc8782404e3b132,5792.0,0.0,2.264276,0.0,1-3,0.003682,-0.003682,CTR_BELOW_POSITION_BENCHMARK,review_meta_title
5,content_76a6fa55e21323b3,5041.0,0.0,2.855291,0.0,1-3,0.003682,-0.003682,CTR_BELOW_POSITION_BENCHMARK,review_meta_title
6,content_9cec93fc44a7ab41,4742.0,0.0,1.852298,0.0,1-3,0.003682,-0.003682,CTR_BELOW_POSITION_BENCHMARK,review_meta_title
7,content_fd1d19e381fc653e,4319.0,0.0,2.789944,0.0,1-3,0.003682,-0.003682,CTR_BELOW_POSITION_BENCHMARK,review_meta_title
8,content_cc24743a90439e14,4225.0,0.0,1.913571,0.0,1-3,0.003682,-0.003682,CTR_BELOW_POSITION_BENCHMARK,review_meta_title
9,content_db01d94616cdb80d,4202.0,0.0,0.911944,0.0,1-3,0.003682,-0.003682,CTR_BELOW_POSITION_BENCHMARK,review_meta_title


## 3. Top-10 review

1. **content_fa17add7** (12,588 impr, pos 1.90) — `review_meta_title`. Highest-volume flag
   in the list; zero clicks at this scale is the strongest possible signal something is wrong.
   Wrong if: a featured snippet or "People Also Ask" box is answering the query directly in
   the SERP, so users never need to click through.
2. **content_d3979871** (9,887 impr, pos 1.95) — `review_meta_title`. Same profile, second
   most confident flag. Wrong if: this is a branded/navigational query where users recognize
   the result but go to the site via a bookmark or app instead of this specific link.
3. **content_a27b382f** (7,736 impr, pos 2.26) — `review_meta_title`. Wrong if: the title/meta
   already looks fine and the real issue is query-intent mismatch, not a fixable snippet problem.
4. **content_83167156** (6,827 impr, pos 1.14) — `review_meta_title`. Best position in the
   entire top 10, still zero clicks — this specific combination is the most suspicious row here.
   Wrong if: this points to a GSC tracking/attribution gap rather than a genuine content issue.
5. **content_1bc87824** (5,792 impr, pos 2.26) — `review_meta_title`. Wrong if: this page's
   traffic comes from a small number of high-volume days rather than being consistent, meaning
   the "average position" hides real day-to-day volatility.
6. **content_76a6fa55** (5,041 impr, pos 2.86) — `review_meta_title`. Wrong if: a competitor's
   rich result (reviews, images) is out-shining this listing even from a nearby position.
7. **content_9cec93fc** (4,742 impr, pos 1.85) — `review_meta_title`. Wrong if: this is a
   duplicate or near-duplicate of another page in the list, and clicks are actually landing on
   the sibling URL instead.
8. **content_fd1d19e3** (4,319 impr, pos 2.79) — `review_meta_title`. Wrong if: this keyword
   is informational-only and users are satisfied by the snippet text alone, with no real need
   to click.
9. **content_cc24743a** (4,225 impr, pos 1.91) — `review_meta_title`. Wrong if: the page was
   only recently promoted to this position and hasn't accumulated clicks yet — a timing issue,
   not a title issue.
10. **content_db01d946** (4,202 impr, pos 0.91) — `review_meta_title`. Best raw position of
    all ten rows, still zero clicks. Wrong if: this is actually a rich-result-only listing
    (e.g. a sitelink or image pack entry) that doesn't behave like a normal organic result.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks / limitations of this baseline

Adding `impressions_march DESC` as a tiebreaker fixed a real problem: without it, ties on
`action_score` were ordered arbitrarily, and re-running the notebook produced a different
top 10 each time. This version is deterministic and prioritizes the highest-confidence
(highest-volume) rows first.

The bigger limitation is that all ten rows share the exact same `action_score`,
`reason_code`, and `action` — this rule can rank *within* the zero-CTR group by volume, but it
can't currently distinguish *why* each one has zero clicks. Ten identical-looking flags with
this much shared volume is also a reason to suspect one systemic cause (a sitewide snippet or
tracking issue) rather than ten independent content problems — worth a human sanity check
before treating these as ten separate title-rewrite tasks.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.